# Feature Engineering Notebook
## Interview Performance Analyzer

**Steps:**
1. Load Data
2. Handle Missing Values
3. Drop Redundant/Low Variance Features
4. NLP Features (TF-IDF + Handcrafted)
5. PCA on MFCC
6. Save Processed Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')

## Step 1: Load Data

In [ ]:
df = pd.read_csv('../../data/processed/clean_master_data.csv')
print(f'Loaded: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'\nColumns: {list(df.columns)}')

## Step 2: Handle Missing Values

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing = missing[missing > 0]
print('Missing values:')
print(missing)

In [ ]:
# Fill transcript missing with empty string
df['transcript'] = df['transcript'].fillna('')

print(f'After handling missing: {df.shape}')
print(f'Missing remaining: {df.isnull().sum().sum()}')

## Step 3: Drop Redundant/Low Variance Features

In [ ]:
# Low variance features (variance < 1e-4)
low_var = ['mouth_frown_mean', 'shoulder_width_var', 
           'nose_shoulder_dist_var', 'core_speed_mean']

# Highly collinear features (|r| > 0.85)
collinear = ['engagement_score', 'agitation_score', 'emotion_happy_mean',
             'emotion_fear_mean', 'emotion_angry_mean', 'emotion_surprise_mean',
             'Spectral_Rolloff', 'gaze_deviation_mean', 'mouth_frown_std',
             'jaw_open_std', 'Articulation_Rate_WPM', 'Pitch_Range_Hz']

drop_features = low_var + collinear
drop_features = [f for f in drop_features if f in df.columns]

print(f'Dropping {len(drop_features)} features:')
print(drop_features)

df = df.drop(columns=drop_features)
print(f'\nAfter dropping: {df.shape}')

## Step 4: NLP Features

In [ ]:
# Handcrafted text features
df['word_count'] = df['transcript'].str.split().str.len()
df['char_count'] = df['transcript'].str.len()
df['avg_word_length'] = df['transcript'].apply(
    lambda x: np.mean([len(w) for w in str(x).split()]) if len(str(x).split()) > 0 else 0
)
df['sentence_count'] = df['transcript'].str.count(r'[.!?]+')
df['unique_word_ratio'] = df['transcript'].apply(
    lambda x: len(set(str(x).split())) / (len(str(x).split()) + 1)
)

# Filler words
fillers = ['um', 'uh', 'like', 'you know', 'basically', 'actually']
df['filler_count'] = df['transcript'].apply(
    lambda x: sum(str(x).lower().count(f) for f in fillers)
)
df['filler_rate'] = df['filler_count'] / (df['word_count'] + 1)
df['question_count'] = df['transcript'].str.count(r'\?')

print(f'Handcrafted features added: {df.shape}')

In [ ]:
# TF-IDF features
tfidf = TfidfVectorizer(
    max_features=200,
    stop_words='english',
    min_df=5,
    max_df=0.95
)

tfidf_matrix = tfidf.fit_transform(df['transcript'].fillna(''))
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=[f'tfidf_{w}' for w in tfidf.get_feature_names_out()]
)

df = pd.concat([df.reset_index(drop=True), tfidf_df.reset_index(drop=True)], axis=1)
print(f'After TF-IDF: {df.shape}')

## Step 5: PCA on MFCC

In [ ]:
# Identify MFCC columns
mfcc_cols = [f'MFCC_{i}' for i in range(1, 14)]
mfcc_cols = [c for c in mfcc_cols if c in df.columns]
print(f'MFCC columns: {len(mfcc_cols)}')

# Apply PCA
pca = PCA(n_components=0.95)  # Retain 95% variance
mfcc_pca = pca.fit_transform(df[mfcc_cols])

n_components = mfcc_pca.shape[1]
print(f'PCA: 13 -> {n_components} components')
print(f'Variance explained: {pca.explained_variance_ratio_.sum():.4f}')

# Create new column names
mfcc_df = pd.DataFrame(
    mfcc_pca,
    columns=[f'MFCC_PC{i+1}' for i in range(n_components)]
)

# Drop original MFCC and add PCA components
df = df.drop(columns=mfcc_cols).reset_index(drop=True)
df = pd.concat([df, mfcc_df.reset_index(drop=True)], axis=1)
print(f'After PCA: {df.shape}')

## Step 6: Save Processed Data

In [ ]:
# Save full dataset
df.to_csv('../../data/processed/feature_engineered_pca.csv', index=False)

# Save numeric features only (for ML)
exclude_cols = ['id', 'file_name', 'duration_label', 'question_id', 
                'question', 'transcript', 'user_no']
target_cols = ['openness', 'conscientiousness', 'extraversion', 
               'agreeableness', 'neuroticism', 'overall_personality',
               'interview_score', 'answer_score', 'speaking_skills',
               'confidence_score', 'facial_expression', 'overall_performance']

feature_cols = [c for c in df.columns if c not in exclude_cols + target_cols]

# Save features and targets separately
df[feature_cols].to_csv('../../data/processed/features_only_pca.csv', index=False)
df[target_cols].to_csv('../../data/processed/targets_only.csv', index=False)

# Save PCA model for deployment
joblib.dump(pca, '../../trained_model/pca.pkl')

print('Saved!')
print(f'\nFeature engineered dataset: {df.shape}')
print(f'Features: {len(feature_cols)}')
print(f'Targets: {len(target_cols)}')
print(f'\nFiles saved:')
print(f'  - feature_engineered_pca.csv')
print(f'  - features_only_pca.csv')
print(f'  - targets_only.csv')
print(f'  - trained_model/pca.pkl')